<a href="https://colab.research.google.com/github/beyzaturku/2209/blob/main/SRCNN_input_hatasi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
import os
import cv2
import h5py
import numpy as np
from pathlib import Path
import gc  # Garbage collector için

# Parametreler
TRAINING_DATA_PATH = "/content/drive/MyDrive/2209/high_reso/train/train"
TEST_DATA_PATH = "/content/drive/MyDrive/2209/high_reso/test/val"
INPUT_SIZE = 32
OUTPUT_SIZE = 20
SCALE = 2

def prepare_data(image_folder, sample_count=None):
    """
    Görüntü klasöründen veri seti hazırlar.
    Hem rastgele hem sistematik örnekleme için tek fonksiyon.
    """
    image_files = sorted(os.listdir(image_folder))
    inputs, targets = [], []

    for filename in image_files:
        # Görüntüyü yükle ve Y kanalını al
        img = cv2.imread(os.path.join(image_folder, filename))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)[:,:,0]

        # Düşük çözünürlüklü versiyon oluştur
        h, w = img.shape
        lr_img = cv2.resize(cv2.resize(img, (w//SCALE, h//SCALE)), (w,h))

        # Görüntüyü parçalara ayır
        for x in range(0, h-INPUT_SIZE, INPUT_SIZE//2):
            for y in range(0, w-INPUT_SIZE, INPUT_SIZE//2):
                lr_patch = lr_img[x:x+INPUT_SIZE, y:y+INPUT_SIZE]
                hr_patch = img[x:x+INPUT_SIZE, y:y+INPUT_SIZE]

                if lr_patch.shape == (INPUT_SIZE,INPUT_SIZE):
                    # Normalize et ve ekle
                    inputs.append(lr_patch[None,:,:,None] / 255.0)
                    targets.append(hr_patch[6:-6,6:-6,None] / 255.0)

        if sample_count and len(inputs) >= sample_count:
            break

    return np.array(inputs, dtype=np.float32), np.array(targets, dtype=np.float32)

def save_h5(inputs, targets, filename):
    """Veriyi HDF5 formatında kaydeder"""
    with h5py.File(filename, 'w') as f:
        f.create_dataset('data', data=inputs)
        f.create_dataset('label', data=targets)

def load_h5(filename):
    """HDF5 dosyasından veri yükler"""
    with h5py.File(filename, 'r') as f:
        return np.array(f['data']), np.array(f['label'])

        data = np.transpose(data, (0, 2, 3, 1))
        labels = np.transpose(labels, (0, 2, 3, 1))
        return data, labels

if __name__ == "__main__":
    # Eğitim verisi hazırla
    print("Eğitim verisi hazırlanıyor...")
    train_in, train_out = prepare_data(TRAINING_DATA_PATH)
    save_h5(train_in, train_out, "/content/drive/MyDrive/2209/high_reso/3_deneme/train.h5")
    print(f"Eğitim verisi: {train_in.shape}")

    # Test verisi hazırla
    print("Test verisi hazırlanıyor...")
    test_in, test_out = prepare_data(TEST_DATA_PATH, sample_count=1000)
    save_h5(test_in, test_out, "/content/drive/MyDrive/2209/high_reso/3_deneme/test.h5")
    print(f"Test verisi: {test_in.shape}")

Eğitim verisi hazırlanıyor...
Eğitim verisi: (1749888, 1, 32, 32, 1)
Test verisi hazırlanıyor...
Test verisi: (1984, 1, 32, 32, 1)


In [3]:
import os
import numpy as np
import math
import cv2
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, Add, Input # Importing Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.optimizers import SGD, Adam
from skimage.metrics import peak_signal_noise_ratio as psnr
#from prepare_random_samples import load_h5_data

def create_training_model():
    """
    Eğitim için geliştirilmiş SRCNN modelini oluşturur.
    Sabit boyutlu giriş şekli (32x32x1) kullanır.
    PSNR performansını artırmak için optimize edilmiştir.
    """
    # Giriş katmanını tanımla - mixed precision kullanımı ekle
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    inputs = Input(shape=(32, 32, 1))

    # İlk özellik çıkarma katmanı
    x = Conv2D(
        filters=64,
        kernel_size=(9, 9),
        kernel_initializer='he_normal',
        padding='same',
        use_bias=True
    )(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    # Orta katman
    x = Conv2D(
        filters=32,
        kernel_size=(5, 5),
        padding='same',
        kernel_initializer='he_normal',
        use_bias=True
    )(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    # Son katman
    x = Conv2D(
        filters=1,
        kernel_size=(5, 5),
        padding='same',
        kernel_initializer='he_normal',
        use_bias=True
    )(x)

    # Modeli oluştur
    model = Model(inputs=inputs, outputs=x)

    # PSNR metriki tanımla
    def psnr(y_true, y_pred):
        return tf.image.psnr(y_true, y_pred, max_val=1.0)

    # Öğrenme oranı scheduler'ı
    initial_learning_rate = 0.001
    decay_steps = 1000
    decay_rate = 0.9
    learning_rate_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate,
        decay_steps=decay_steps,
        decay_rate=decay_rate,
        staircase=True
    )

    # Callback'leri tanımla
    early_stopping = EarlyStopping(
        monitor='val_psnr',
        mode='max',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )

    model_checkpoint = ModelCheckpoint(
        filepath='best_model.h5',
        monitor='val_psnr',
        mode='max',
        save_best_only=True,
        verbose=1
    )

    # Model derleme
    adam_optimizer = Adam(learning_rate=learning_rate_schedule)
    model.compile(
        optimizer=adam_optimizer,
        loss='mean_squared_error',
        metrics=['mean_squared_error', psnr],
        run_eagerly=False
    )

    # Callback'leri model özelliğine ekle
    model.callbacks = [early_stopping, model_checkpoint]

    # Memory cleanup için garbage collector ekle
    import gc
    gc.collect()
    tf.keras.backend.clear_session()

    return model

In [4]:
def create_prediction_model():
    """
    Tahmin için SRCNN modelini oluşturur.
    Değişken boyutlu giriş şekli (None, None, 1) kullanır.
    """
     # Giriş katmanını tanımla - mixed precision kullanımı ekle
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    inputs = Input(shape=(None, None, 1))

    # İlk özellik çıkarma katmanı
    x = Conv2D(
        filters=64,
        kernel_size=(9, 9),
        kernel_initializer='he_normal',
        padding='same',
        use_bias=True
    )(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    # Orta katman
    x = Conv2D(
        filters=32,
        kernel_size=(5, 5),
        padding='same',
        kernel_initializer='he_normal',
        use_bias=True
    )(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    # Son katman
    x = Conv2D(
        filters=1,
        kernel_size=(5, 5),
        padding='same',
        kernel_initializer='he_normal',
        use_bias=True
    )(x)

    # Modeli oluştur
    model = Model(inputs=inputs, outputs=x)

    # PSNR metriki tanımla
    def psnr(y_true, y_pred):
        return tf.image.psnr(y_true, y_pred, max_val=1.0)

    # Öğrenme oranı scheduler'ı
    initial_learning_rate = 0.001
    decay_steps = 1000
    decay_rate = 0.9
    learning_rate_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate,
        decay_steps=decay_steps,
        decay_rate=decay_rate,
        staircase=True
    )

    # Callback'leri tanımla
    early_stopping = EarlyStopping(
        monitor='val_psnr',
        mode='max',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )

    model_checkpoint = ModelCheckpoint(
        filepath='best_model.h5',
        monitor='val_psnr',
        mode='max',
        save_best_only=True,
        verbose=1
    )

    # Model derleme
    adam_optimizer = Adam(learning_rate=learning_rate_schedule)
    model.compile(
        optimizer=adam_optimizer,
        loss='mean_squared_error',
        metrics=['mean_squared_error', psnr],
        run_eagerly=False
    )

    # Callback'leri model özelliğine ekle
    model.callbacks = [early_stopping, model_checkpoint]

    # Memory cleanup için garbage collector ekle
    import gc
    gc.collect()
    tf.keras.backend.clear_session()

    return model

In [5]:
def calculate_psnr(img1, img2):
    """
    İki görüntü arasındaki PSNR'yi (Peak Signal-to-Noise Ratio) hesaplar.
    """
    mse = np.mean((img1 - img2)**2)
    if mse == 0:
        return float('inf')  # MSE 0 ise PSNR sonsuzdur
    max_pixel = 255.0
    psnr = 20 * math.log10(max_pixel / math.sqrt(mse))
    return psnr

In [15]:
def train_model():
    """
    SRCNN modelini eğitir ve en iyi modeli kaydeder.
    """
    # Eğitim modelini oluştur
    srcnn_model = create_training_model()
    print(srcnn_model.summary())

    # Eğitim ve doğrulama verilerini yükle
    print("Eğitim verilerini yükleme...")
    train_data, train_labels = load_h5("/content/drive/MyDrive/2209/high_reso/3_deneme/train.h5")
    val_data, val_labels = load_h5("/content/drive/MyDrive/2209/high_reso/3_deneme/test.h5")

    train_data = np.transpose(train_data, (0, 2, 3, 1))
    val_data = np.transpose(val_data, (0, 2, 3, 1))

    # Model kaydetme için callback oluştur
    checkpoint = ModelCheckpoint(
        "SRCNN.h5",
        monitor='val_loss',
        verbose=1,
        save_best_only=True,
        save_weights_only=False,
        mode='min'
    )
    callbacks_list = [checkpoint]

    # Modeli eğit
    print("Model eğitimi başlıyor...")
    srcnn_model.fit(
        train_data, train_labels,
        batch_size=16,
        validation_data=(val_data, val_labels),
        callbacks=callbacks_list,
        shuffle=True,
        epochs=300,
        verbose=1
    )

    print("Eğitim tamamlandı!")

In [18]:
if __name__ == "__main__":
    # Modeli eğit
    train_model()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 32, 32, 1)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ cast (Cast)                          │ (None, 32, 32, 1)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d (Conv2D)                      │ (None, 32, 32, 64)          │           5,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 32, 32, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation (Activation)              │ (None, 32, 32, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 32, 32, 32)          │          51,232 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 32, 32, 32)          │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_1 (Activation)            │ (None, 32, 32, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 32, 32, 1)           │             801 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 57,665 (225.25 KB)

 Trainable params: 57,473 (224.50 KB)

 Non-trainable params: 192 (768.00 B)

None
Eğitim verilerini yükleme...
train_data shape: (1749888, 1, 32, 32, 1)
val_data shape: (1984, 1, 32, 32, 1)
Model eğitimi başlıyor...
Epoch 1/300


ValueError: Input 0 of layer "functional" is incompatible with the layer: expected shape=(None, 32, 32, 1), found shape=(16, 1, 32, 32)

In [14]:
def predict_image(model_path, image_path, output_folder="/content/drive/MyDrive/2209/high_reso/results"):
    """
    Bir görüntüyü SRCNN ile süper çözünürlüklü hale getirir.

        model_path: Eğitilmiş model ağırlıklarının yolu
        image_path: Girdi görüntüsünün yolu
        output_folder: Sonuçların kaydedileceği klasör
    """
    # Çıktı klasörünü oluştur
    os.makedirs(output_folder, exist_ok=True)

    # Dosya adı bilgilerini hazırla
    base_name = os.path.basename(image_path)
    file_name, _ = os.path.splitext(base_name)
    input_path = os.path.join(output_folder, f"{file_name}_bicubic.png")
    output_path = os.path.join(output_folder, f"{file_name}_srcnn.png")

    # Tahmin modelini oluştur ve ağırlıkları yükle
    srcnn_model = create_prediction_model()
    srcnn_model.load_weights(model_path)
    print(f"Model yüklendi: {model_path}")

    # Orijinal görüntüyü yükle
    img = cv2.imread(image_path, cv2.IMREAD_COLOR)

    # BGR'dan YCrCb'ye dönüştür
    img_ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)
    height, width = img_ycrcb.shape[:2]

    # Y kanalını önce küçült sonra bicubic ile büyüt (düşük çözünürlük simulasyonu)
    y_channel = img_ycrcb[:, :, 0]
    y_channel_lr = cv2.resize(y_channel, (width // 2, height // 2), cv2.INTER_CUBIC)
    y_channel_bicubic = cv2.resize(y_channel_lr, (width, height), cv2.INTER_CUBIC)

    # Bicubic sonucunu kaydet
    img_bicubic = img_ycrcb.copy()
    img_bicubic[:, :, 0] = y_channel_bicubic
    img_bicubic = cv2.cvtColor(img_bicubic, cv2.COLOR_YCrCb2BGR)
    cv2.imwrite(input_path, img_bicubic)
    print(f"Bicubic upscaled görüntü kaydedildi: {input_path}")

    # SRCNN için girdiyi hazırla
    input_data = np.zeros((1, height, width, 1), dtype=float)
    input_data[0, :, :, 0] = y_channel_bicubic.astype(float) / 255.0

    # SRCNN tahmini yap
    prediction = srcnn_model.predict(input_data, batch_size=1) * 255.0

    # Değerleri [0, 255] aralığına kırp
    prediction = np.clip(prediction, 0, 255).astype(np.uint8)

    # Tahmini orijinal görüntüye yerleştir (evrişim padding nedeniyle 6 piksel kenarları kırpılır)
    img_srcnn = img_ycrcb.copy()
    img_srcnn[6:-6, 6:-6, 0] = prediction[0, :, :, 0]
    img_srcnn = cv2.cvtColor(img_srcnn, cv2.COLOR_YCrCb2BGR)
    cv2.imwrite(output_path, img_srcnn)
    print(f"SRCNN sonucu kaydedildi: {output_path}")

    # Değerlendirme için görüntüleri hazırla
    original = img[6:-6, 6:-6]
    bicubic = img_bicubic[6:-6, 6:-6]
    srcnn = img_srcnn[6:-6, 6:-6]

    # Y kanallarını hazırla
    original_y = cv2.cvtColor(original, cv2.COLOR_BGR2YCrCb)[:, :, 0]
    bicubic_y = cv2.cvtColor(bicubic, cv2.COLOR_BGR2YCrCb)[:, :, 0]
    srcnn_y = cv2.cvtColor(srcnn, cv2.COLOR_BGR2YCrCb)[:, :, 0]

    # PSNR hesaplama
    original_y = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)[6:-6, 6:-6, 0]
    bicubic_y = cv2.cvtColor(img_bicubic, cv2.COLOR_BGR2YCrCb)[6:-6, 6:-6, 0]
    srcnn_y = cv2.cvtColor(img_srcnn, cv2.COLOR_BGR2YCrCb)[6:-6, 6:-6, 0]

    bicubic_psnr = calculate_psnr(original_y, bicubic_y)
    srcnn_psnr = calculate_psnr(original_y, srcnn_y)

    # SSIM hesaplama
    bicubic_ssim = structural_similarity(original_y, bicubic_y)
    srcnn_ssim = structural_similarity(original_y, srcnn_y)

    # RMSE hesaplama
    bicubic_rmse = np.sqrt(np.mean((original_y - bicubic_y) ** 2))
    srcnn_rmse = np.sqrt(np.mean((original_y - srcnn_y) ** 2))

    # LPIPS hesaplama (RGB görüntüler için)
    loss_fn = lpips.LPIPS(net='alex')
    bicubic_lpips = loss_fn(transforms.ToTensor()(bicubic), transforms.ToTensor()(original)).item()
    srcnn_lpips = loss_fn(transforms.ToTensor()(srcnn), transforms.ToTensor()(original)).item()

    # NIQE hesaplama
    bicubic_niqe = niqe(bicubic_y)
    srcnn_niqe = niqe(srcnn_y)

    # BRISQUE hesaplama
    bicubic_brisque = brisque.score(bicubic)
    srcnn_brisque = brisque.score(srcnn)

    # Sonuçları yazdır
    print("\nGörüntü Kalite Metrikleri:")
    print(f"PSNR (dB) - Bicubic: {bicubic_psnr:.2f}, SRCNN: {srcnn_psnr:.2f}, İyileşme: {srcnn_psnr - bicubic_psnr:.2f}")
    print(f"SSIM - Bicubic: {bicubic_ssim:.4f}, SRCNN: {srcnn_ssim:.4f}, İyileşme: {srcnn_ssim - bicubic_ssim:.4f}")
    print(f"RMSE - Bicubic: {bicubic_rmse:.4f}, SRCNN: {srcnn_rmse:.4f}, İyileşme: {bicubic_rmse - srcnn_rmse:.4f}")
    print(f"LPIPS - Bicubic: {bicubic_lpips:.4f}, SRCNN: {srcnn_lpips:.4f}, İyileşme: {bicubic_lpips - srcnn_lpips:.4f}")
    print(f"NIQE - Bicubic: {bicubic_niqe:.4f}, SRCNN: {srcnn_niqe:.4f}, İyileşme: {bicubic_niqe - srcnn_niqe:.4f}")
    print(f"BRISQUE - Bicubic: {bicubic_brisque:.4f}, SRCNN: {srcnn_brisque:.4f}, İyileşme: {bicubic_brisque - srcnn_brisque:.4f}")

    # MOS için not: MOS (Mean Opinion Score) öznel bir değerlendirme metriğidir
    # ve insan değerlendirmesi gerektirir. Otomatik olarak hesaplanamaz.
    print("\nNot: MOS (Mean Opinion Score) öznel bir değerlendirme metriğidir ve insan değerlendirmesi gerektirir.")

In [ ]:
predict_image(
        model_path="/content/SRCNN.h5",
        image_path="/content/drive/MyDrive/2209/high_reso/test/val/M0501_img000010.jpg",
        output_folder="/content/drive/MyDrive/2209/high_reso/results"
    )

Model yüklendi: /content/SRCNN.h5
Bicubic upscaled görüntü kaydedildi: /content/drive/MyDrive/2209/high_reso/results/M0501_img000010_bicubic.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
SRCNN sonucu kaydedildi: /content/drive/MyDrive/2209/high_reso/results/M0501_img000010_srcnn.png
Bicubic PSNR: 33.28 dB
SRCNN PSNR: 34.02 dB
PSNR İyileştirmesi: 0.74 dB
